# `tbl-datasets`: training, validation, and held-out test datasets

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "tbl-datasets"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/tbl-datasets.*


## Labels and formatting (edit wording here)

In [2]:
ma.setup_analysis_env()
from beta_ols import berke2023 as ds_berke
from beta_ols import covid as ds_covid
from beta_ols import grummon2025 as ds_grummon
from beta_ols import kantorowicz2026 as ds_kantorowicz
from beta_ols import kriner2026 as ds_kriner
from beta_ols import magistro2025 as ds_magistro
from beta_ols import wu2025 as ds_wu

HEADERS = ["Role", "Dataset", "Price attributes", "Non-price attributes", "Respondents", "Tasks", "$d$"]
# Pandoc sets relative column widths from the dash counts.
COLUMN_RULES = [":---", ":-----", ":-------------", ":-------------", "---:", "---:", "---:"]

# One row each, in this order: (data module, respondent id column, role, dataset label, citation key, price text, non-price text).
DATASET_ROWS = [
    (ds_wu, "CaseID", "Training", "Electricity", "wu2025partisan", "Monthly bill ($105 to $125)", "Renewable share (10% to 100%), local renewables"),
    (ds_berke, "ResponseId", "Training", "Drones", "berke2023drone", "Delivery fee ($0 to $10)", "Wait time, drone privacy"),
    (ds_kriner, "respondent", "Validation", "Flights", "goldfarb2026comfort", "Ticket price ($250 to $325)", "Sustainable aviation fuel, airline, baggage, legroom, seat"),
    (ds_kantorowicz, "CASEID", "Test", "Consumer rights", "kantorowicz2026consumers", "Laptop ($500 to $2,000) and laundry ($400 to $1,400) price", "Brand, specs, warranty, privacy, liability, conflict resolution"),
    (ds_magistro, "CASEID", "Test", "AI & offshoring", "magistro2026attitudes", "Smartphone price ($300 to $600)", "Customer service, factory, and data science jobs"),
    (ds_grummon, "participantid", "Test", "Food labels", "grummon2025nutrition", "None", "Front-of-package warning labels (6 arms)"),
    (ds_covid, "respondent", "Test", "Vaccines", "kreps2020factors", "None", "Efficacy, duration, side effects, FDA, origin, endorsements"),
]

## Build the table (counts come from the data)

In [3]:
def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table: header row, rule row (e.g. ':---' left, '---:' right), then one line per row."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(column_rules) + "|")
    for row in rows:
        if len(row) != len(headers):
            raise ValueError(f"Row has {len(row)} cells, expected {len(headers)}: {row}")
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

In [4]:
rows = []
for module, respondent_column, role, dataset_label, citation_key, price_text, nonprice_text in DATASET_ROWS:
    choice_df = module.load_choice_df()
    n_respondents = int(choice_df[respondent_column].nunique())
    n_tasks = int(len(choice_df))
    specs = module.regression_specs(module.regression_rows(choice_df))
    n_slopes = 0
    for spec in specs.values():
        n_slopes += len(spec.target_labels)
    rows.append([
        role,
        f"{dataset_label} [@{citation_key}]",
        escape_markdown(price_text),
        escape_markdown(nonprice_text),
        f"{n_respondents:,}",
        f"{n_tasks:,}",
        f"{n_slopes}",
    ])

table_md = pipe_table(HEADERS, rows, COLUMN_RULES)
display(pd.DataFrame(rows, columns=HEADERS))

,Role,Dataset,Price attributes,Non-price attributes,Respondents,Tasks,$d$
0,Training,Electricity [@wu2025partisan],Monthly bill (\$105 to \$125),"Renewable share (10% to 100%), local renewables","1,861","18,591",8
1,Training,Drones [@berke2023drone],Delivery fee (\$0 to \$10),"Wait time, drone privacy","3,625","36,297",28
2,Validation,Flights [@goldfarb2026comfort],Ticket price (\$250 to \$325),"Sustainable aviation fuel, airline, baggage, l...","1,000","5,000",12
3,Test,Consumer rights [@kantorowicz2026consumers],"Laptop (\$500 to \$2,000) and laundry (\$400 t...","Brand, specs, warranty, privacy, liability, co...","2,242","26,823",38
4,Test,AI & offshoring [@magistro2026attitudes],Smartphone price (\$300 to \$600),"Customer service, factory, and data science jobs","5,525","21,005",32
5,Test,Food labels [@grummon2025nutrition],None,Front-of-package warning labels (6 arms),"5,633","33,798",5
6,Test,Vaccines [@kreps2020factors],None,"Efficacy, duration, side effects, FDA, origin,...","1,971","9,855",11


## Save `tbl-datasets.md` (included by the manuscript)

In [5]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Role | Dataset | Price attributes | Non-price attributes | Respondents | Tasks | $d$ |
|:---|:-----|:-------------|:-------------|---:|---:|---:|
| Training | Electricity [@wu2025partisan] | Monthly bill (\$105 to \$125) | Renewable share (10% to 100%), local renewables | 1,861 | 18,591 | 8 |
| Training | Drones [@berke2023drone] | Delivery fee (\$0 to \$10) | Wait time, drone privacy | 3,625 | 36,297 | 28 |
| Validation | Flights [@goldfarb2026comfort] | Ticket price (\$250 to \$325) | Sustainable aviation fuel, airline, baggage, legroom, seat | 1,000 | 5,000 | 12 |
| Test | Consumer rights [@kantorowicz2026consumers] | Laptop (\$500 to \$2,000) and laundry (\$400 to \$1,400) price | Brand, specs, warranty, privacy, liability, conflict resolution | 2,242 | 26,823 | 38 |
| Test | AI & offshoring [@magistro2026attitudes] | Smartphone price (\$300 to \$600) | Customer service, factory, and data science jobs | 5,525 | 21,005 | 32 |
| Test | Food labels [@grummon2025nutrition] | None | F